In [1]:
"""
delta_optimizer.py
==================
Ottimizzazione tabelle Delta Lake in base al livello Medallion Architecture
e allo scopo di visualizzazione del dato.

Compatibile con: Microsoft Fabric (Spark), Databricks

Autore: Marco Pozzan
"""

from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("DeltaOptimizer")


# ─────────────────────────────────────────────
#  ENUM: Livelli Medallion
# ─────────────────────────────────────────────
class MedallionLayer(str, Enum):
    BRONZE = "bronze"   # Raw / Landing
    SILVER = "silver"   # Cleansed / Conformed
    GOLD   = "gold"     # Aggregated / Serving


# ─────────────────────────────────────────────
#  ENUM: Scopo di visualizzazione
# ─────────────────────────────────────────────
class VisualizationPurpose(str, Enum):
    OPERATIONAL    = "operational"    # Report operativi real-time / near real-time
    ANALYTICAL     = "analytical"     # Analisi storiche, trend, drill-down
    DASHBOARD_KPI  = "dashboard_kpi"  # KPI su Power BI / Direct Lake (letture aggregate veloci)
    EXPORT         = "export"         # Export bulk verso altri sistemi
    ML_TRAINING    = "ml_training"    # Feature store per modelli ML
    ARCHIVE        = "archive"        # Storico freddo, accesso raro


# ─────────────────────────────────────────────
#  PROFILO DI OTTIMIZZAZIONE
# ─────────────────────────────────────────────
@dataclass
class OptimizationProfile:
    # OPTIMIZE / Z-ORDER
    run_optimize:       bool = True
    zorder_columns:     list[str] = field(default_factory=list)

    # VACUUM
    run_vacuum:         bool = True
    vacuum_retain_hours: int = 168          # default 7 giorni

    # AUTO OPTIMIZE (Fabric / Databricks)
    auto_optimize:      bool = False        # optimizeWrite
    auto_compact:       bool = False        # autoCompact

    # TARGET FILE SIZE
    target_file_size_mb: Optional[int] = None  # None = usa default Delta

    # PARTITION PRUNING hint
    partition_hint:     Optional[str] = None

    # DATA SKIPPING
    data_skipping_columns: list[str] = field(default_factory=list)

    # STATS COLUMNS (Delta stats collection)
    stats_columns:      int = 32            # delta.dataSkippingNumIndexedCols

    # DELETION VECTORS (Databricks / Fabric)
    enable_deletion_vectors: bool = False

    # LIQUID CLUSTERING (Databricks / Fabric preview)
    enable_liquid_clustering: bool = False
    liquid_cluster_columns:   list[str] = field(default_factory=list)

    # NOTE descrittive
    notes: str = ""


# ─────────────────────────────────────────────
#  MATRICE DI STRATEGIA
# ─────────────────────────────────────────────
def get_optimization_profile(
    layer: MedallionLayer,
    purpose: VisualizationPurpose,
    zorder_columns: Optional[list[str]] = None,
    partition_column: Optional[str] = None,
) -> OptimizationProfile:
    """
    Restituisce un OptimizationProfile configurato in base a layer + purpose.
    zorder_columns: colonne suggerite per Z-ORDER (es. ["DataOrdine", "ClienteID"])
    partition_column: colonna di partizione principale della tabella
    """

    zo = zorder_columns or []
    p = OptimizationProfile()

    # ── BRONZE ──────────────────────────────────────────────────────────────
    if layer == MedallionLayer.BRONZE:

        if purpose == VisualizationPurpose.ARCHIVE:
            p.run_optimize        = False   # Raw non ottimizzato, solo retention
            p.vacuum_retain_hours = 720     # 30 giorni — audit trail
            p.auto_optimize       = False
            p.target_file_size_mb = 256     # file grandi, poche letture
            p.notes = "Bronze/Archive: nessun OPTIMIZE, retention lunga per audit."

        else:  # operativo, export, ecc.
            p.run_optimize        = True
            p.zorder_columns      = zo
            p.vacuum_retain_hours = 336     # 14 giorni
            p.auto_optimize       = True    # scrivi file ottimizzati già in ingestion
            p.auto_compact        = False
            p.target_file_size_mb = 128
            p.notes = "Bronze: OPTIMIZE + autoOptimizeWrite per ridurre small files in ingestion."

    # ── SILVER ──────────────────────────────────────────────────────────────
    elif layer == MedallionLayer.SILVER:

        if purpose == VisualizationPurpose.OPERATIONAL:
            p.run_optimize        = True
            p.zorder_columns      = zo      # es. ["EventTimestamp", "DeviceID"]
            p.vacuum_retain_hours = 168
            p.auto_optimize       = True
            p.auto_compact        = True
            p.target_file_size_mb = 64      # file piccoli → latenza bassa
            p.enable_deletion_vectors = True
            p.notes = "Silver/Operational: file piccoli, DV abilitati per UPSERT veloci."

        elif purpose == VisualizationPurpose.ANALYTICAL:
            p.run_optimize        = True
            p.zorder_columns      = zo      # es. ["Anno", "CodiceCliente"]
            p.vacuum_retain_hours = 336
            p.auto_optimize       = True
            p.auto_compact        = True
            p.target_file_size_mb = 256
            p.stats_columns       = 32
            p.notes = "Silver/Analytical: file medi, Z-ORDER su colonne di filtro frequente."

        elif purpose == VisualizationPurpose.ML_TRAINING:
            p.run_optimize        = True
            p.zorder_columns      = []      # ML legge tutto, Z-ORDER poco utile
            p.vacuum_retain_hours = 168
            p.auto_optimize       = True
            p.auto_compact        = True
            p.target_file_size_mb = 512     # file grandi → throughput lettura massima
            p.stats_columns       = 8       # meno stats overhead
            p.notes = "Silver/ML: file grandi per massimizzare throughput scan sequenziale."

        else:
            p.run_optimize        = True
            p.zorder_columns      = zo
            p.vacuum_retain_hours = 168
            p.auto_optimize       = True
            p.notes = "Silver: profilo generico bilanciato."

    # ── GOLD ────────────────────────────────────────────────────────────────
    elif layer == MedallionLayer.GOLD:

        if purpose == VisualizationPurpose.DASHBOARD_KPI:
            p.run_optimize        = True
            p.zorder_columns      = zo      # es. ["Anno", "Mese", "BU"]
            p.vacuum_retain_hours = 168
            p.auto_optimize       = True
            p.auto_compact        = True
            p.target_file_size_mb = 128     # bilanciato per Direct Lake framing
            p.enable_deletion_vectors = True
            p.stats_columns       = 32
            # Liquid clustering preferito su Gold KPI se Fabric/Databricks supporta
            if zo:
                p.enable_liquid_clustering = True
                p.liquid_cluster_columns   = zo[:2]  # max 2-4 colonne cluster
            p.notes = (
                "Gold/Dashboard KPI: Liquid Clustering + DV per Direct Lake ottimale. "
                "Z-ORDER su dimensioni usate nei filtri Power BI."
            )

        elif purpose == VisualizationPurpose.ANALYTICAL:
            p.run_optimize        = True
            p.zorder_columns      = zo
            p.vacuum_retain_hours = 336
            p.auto_optimize       = True
            p.auto_compact        = True
            p.target_file_size_mb = 256
            p.stats_columns       = 32
            p.notes = "Gold/Analytical: file medi-grandi, Z-ORDER su chiavi analitiche."

        elif purpose == VisualizationPurpose.EXPORT:
            p.run_optimize        = True
            p.zorder_columns      = []
            p.vacuum_retain_hours = 72      # breve, dato che viene ricreato spesso
            p.auto_optimize       = False
            p.target_file_size_mb = 512     # massimizza throughput export
            p.stats_columns       = 4
            p.notes = "Gold/Export: file grandi, nessun Z-ORDER, retention corta."

        elif purpose == VisualizationPurpose.OPERATIONAL:
            p.run_optimize        = True
            p.zorder_columns      = zo
            p.vacuum_retain_hours = 72
            p.auto_optimize       = True
            p.auto_compact        = True
            p.target_file_size_mb = 64
            p.enable_deletion_vectors = True
            p.notes = "Gold/Operational: tabelle aggregate con aggiornamenti frequenti, DV abilitati."

        elif purpose == VisualizationPurpose.ARCHIVE:
            p.run_optimize        = False
            p.run_vacuum          = True
            p.vacuum_retain_hours = 2160    # 90 giorni
            p.target_file_size_mb = 1024
            p.notes = "Gold/Archive: nessun OPTIMIZE, file grandi, retention massima."

        else:
            p.run_optimize        = True
            p.zorder_columns      = zo
            p.vacuum_retain_hours = 168
            p.auto_optimize       = True
            p.notes = "Gold: profilo generico serving layer."

    p.partition_hint = partition_column
    return p


# ─────────────────────────────────────────────
#  APPLICAZIONE DEL PROFILO
# ─────────────────────────────────────────────
def apply_optimization(
    spark: SparkSession,
    table_path: str,
    profile: OptimizationProfile,
    dry_run: bool = False,
) -> dict:
    """
    Applica il profilo di ottimizzazione a una tabella Delta.

    Parameters
    ----------
    spark      : SparkSession attiva
    table_path : path ABFSS o nome tabella Hive (es. "bronze.raw_events")
    profile    : OptimizationProfile calcolato
    dry_run    : se True stampa le operazioni senza eseguirle

    Returns
    -------
    dict con riepilogo operazioni eseguite
    """

    ops = []

    log.info(f"{'[DRY RUN] ' if dry_run else ''}Ottimizzazione: {table_path}")
    log.info(f"  Profilo note: {profile.notes}")

    # ── Configura proprietà tabella Delta ──────────────────────────────────
    props = {}

    if profile.target_file_size_mb:
        props["delta.targetFileSize"] = str(profile.target_file_size_mb * 1024 * 1024)

    if profile.auto_optimize:
        props["delta.autoOptimize.optimizeWrite"] = "true"
    if profile.auto_compact:
        props["delta.autoOptimize.autoCompact"] = "true"
    if profile.enable_deletion_vectors:
        props["delta.enableDeletionVectors"] = "true"
    if profile.stats_columns:
        props["delta.dataSkippingNumIndexedCols"] = str(profile.stats_columns)

    if props and not dry_run:
        alter_sql = ", ".join([f"'{k}' = '{v}'" for k, v in props.items()])
        spark.sql(f"ALTER TABLE delta.`{table_path}` SET TBLPROPERTIES ({alter_sql})")
        ops.append(f"SET TBLPROPERTIES: {props}")
        log.info(f"  SET TBLPROPERTIES: {props}")
    elif props:
        log.info(f"  [DRY RUN] SET TBLPROPERTIES: {props}")
        ops.append(f"[DRY RUN] SET TBLPROPERTIES: {props}")

    # ── Liquid Clustering ──────────────────────────────────────────────────
    if profile.enable_liquid_clustering and profile.liquid_cluster_columns:
        cluster_cols = ", ".join(profile.liquid_cluster_columns)
        sql = f"ALTER TABLE delta.`{table_path}` CLUSTER BY ({cluster_cols})"
        log.info(f"  Liquid Clustering: {cluster_cols}")
        if not dry_run:
            spark.sql(sql)
        ops.append(f"CLUSTER BY ({cluster_cols})")

    # ── OPTIMIZE ───────────────────────────────────────────────────────────
    if profile.run_optimize:
        if profile.zorder_columns:
            zorder_str = ", ".join(profile.zorder_columns)
            sql = f"OPTIMIZE delta.`{table_path}` ZORDER BY ({zorder_str})"
            log.info(f"  OPTIMIZE ZORDER BY ({zorder_str})")
        else:
            sql = f"OPTIMIZE delta.`{table_path}`"
            log.info("  OPTIMIZE (no Z-ORDER)")

        if not dry_run:
            spark.sql(sql)
        ops.append(sql)

    # ── VACUUM ─────────────────────────────────────────────────────────────
    if profile.run_vacuum:
        sql = f"VACUUM delta.`{table_path}` RETAIN {profile.vacuum_retain_hours} HOURS"
        log.info(f"  VACUUM RETAIN {profile.vacuum_retain_hours} HOURS")
        if not dry_run:
            spark.sql(sql)
        ops.append(sql)

    log.info("  ✓ Completato.\n")
    return {"table": table_path, "operations": ops, "dry_run": dry_run}


# ─────────────────────────────────────────────
#  VERIFICA PROPRIETÀ APPLICATE
# ─────────────────────────────────────────────

# Mappa proprietà Delta → valore atteso dal profilo
PROP_MAP = {
    "delta.targetFileSize":                  lambda p: str(p.target_file_size_mb * 1024 * 1024) if p.target_file_size_mb else None,
    "delta.autoOptimize.optimizeWrite":      lambda p: "true" if p.auto_optimize else None,
    "delta.autoOptimize.autoCompact":        lambda p: "true" if p.auto_compact else None,
    "delta.enableDeletionVectors":           lambda p: "true" if p.enable_deletion_vectors else None,
    "delta.dataSkippingNumIndexedCols":      lambda p: str(p.stats_columns) if p.stats_columns else None,
}

@dataclass
class PropertyCheckResult:
    prop:     str
    expected: str
    actual:   Optional[str]

    @property
    def ok(self) -> bool:
        return self.actual == self.expected

    def __str__(self) -> str:
        status = "✅" if self.ok else "❌"
        actual_display = self.actual if self.actual is not None else "<non impostata>"
        return f"  {status} {self.prop}: atteso={self.expected}  effettivo={actual_display}"


def verify_table_properties(
    spark: SparkSession,
    table_path: str,
    profile: OptimizationProfile,
) -> list[PropertyCheckResult]:
    """
    Legge le TBLPROPERTIES effettive della tabella Delta e le confronta
    con i valori attesi dal profilo di ottimizzazione.

    Returns
    -------
    Lista di PropertyCheckResult con esito per ogni proprietà.
    """

    # Leggi le proprietà effettive dalla tabella
    try:
        rows = spark.sql(f"SHOW TBLPROPERTIES delta.`{table_path}`").collect()
        actual_props = {row["key"]: row["value"] for row in rows}
    except Exception as e:
        log.error(f"  Impossibile leggere TBLPROPERTIES per {table_path}: {e}")
        return []

    results = []
    for prop_key, expected_fn in PROP_MAP.items():
        expected_val = expected_fn(profile)
        if expected_val is None:
            continue  # proprietà non richiesta da questo profilo, skip

        actual_val = actual_props.get(prop_key)
        result = PropertyCheckResult(
            prop=prop_key,
            expected=expected_val,
            actual=actual_val,
        )
        results.append(result)

        if result.ok:
            log.info(str(result))
        else:
            log.warning(str(result))

    # Verifica Liquid Clustering (tramite DESCRIBE DETAIL)
    if profile.enable_liquid_clustering and profile.liquid_cluster_columns:
        try:
            detail = spark.sql(f"DESCRIBE DETAIL delta.`{table_path}`").collect()[0]
            cluster_cols_actual = detail["clusteringColumns"] or []
            cluster_cols_expected = profile.liquid_cluster_columns

            ok = sorted(cluster_cols_actual) == sorted(cluster_cols_expected)
            status = "OK" if ok else "KO"
            log.info(
                f"  {status} clusteringColumns: "
                f"atteso={cluster_cols_expected}  effettivo={cluster_cols_actual}"
            )
            results.append(PropertyCheckResult(
                prop="clusteringColumns",
                expected=str(sorted(cluster_cols_expected)),
                actual=str(sorted(cluster_cols_actual)),
            ))
        except Exception as e:
            log.warning(f"    Impossibile verificare Liquid Clustering: {e}")

    return results


def verify_and_report(
    spark: SparkSession,
    table_path: str,
    profile: OptimizationProfile,
) -> dict:
    """
    Esegue la verifica e restituisce un dizionario riepilogativo con:
        - table        : nome tabella
        - checks       : lista PropertyCheckResult
        - passed       : numero check superati
        - failed       : numero check falliti
        - success      : True se tutti i check sono passati
    """
    log.info(f"\n Verifica proprietà: {table_path}")
    checks = verify_table_properties(spark, table_path, profile)

    passed = sum(1 for c in checks if c.ok)
    failed = sum(1 for c in checks if not c.ok)

    summary = {
        "table":   table_path,
        "checks":  checks,
        "passed":  passed,
        "failed":  failed,
        "success": failed == 0,
    }

    if failed == 0:
        log.info(f"   Tutti i {passed} check superati.")
    else:
        log.warning(f"    {failed} check falliti su {passed + failed} totali.")

    return summary


# ─────────────────────────────────────────────
#  HELPER: ottimizza lista di tabelle
# ─────────────────────────────────────────────
def optimize_tables(
    spark: SparkSession,
    tables: list[dict],
    dry_run: bool = False,
    verify: bool = True,
) -> list[dict]:
    """
    Ottimizza una lista di tabelle con configurazione per ognuna.
    Se verify=True (default), esegue la verifica delle proprietà dopo ogni ottimizzazione.

    Ogni elemento di `tables` è un dict con:
        - path          (str)  : path o nome tabella
        - layer         (MedallionLayer)
        - purpose       (VisualizationPurpose)
        - zorder_cols   (list[str], opzionale)
        - partition_col (str, opzionale)

    Esempio:
        tables = [
            {
                "path": "abfss://silver@storage.dfs.core.windows.net/vendite",
                "layer": MedallionLayer.SILVER,
                "purpose": VisualizationPurpose.ANALYTICAL,
                "zorder_cols": ["Anno", "CodiceCliente"],
            },
            ...
        ]
    """
    results = []
    for t in tables:
        profile = get_optimization_profile(
            layer=t["layer"],
            purpose=t["purpose"],
            zorder_columns=t.get("zorder_cols"),
            partition_column=t.get("partition_col"),
        )
        result = apply_optimization(spark, t["path"], profile, dry_run=dry_run)

        # Verifica post-ottimizzazione (solo se non dry_run)
        if verify and not dry_run:
            result["verification"] = verify_and_report(spark, t["path"], profile)
        else:
            result["verification"] = None

        results.append(result)
    return results


# ─────────────────────────────────────────────
#  ESEMPIO D'USO
# ─────────────────────────────────────────────
if __name__ == "__main__":

    spark = SparkSession.builder.appName("DeltaOptimizer").getOrCreate()

    # Disabilita il check della retention per ambienti di test
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

    tables_to_optimize = [
        # Bronze raw ingestion
        {
            "path": "abfss://2cd8c8d6-210e-453e-a27d-453b9ffc21f1@onelake.dfs.fabric.microsoft.com/c3fcb4c1-59fb-408a-9972-b35608d80496/Tables/DB_Countries",
            "layer": MedallionLayer.BRONZE,
            "purpose": VisualizationPurpose.ARCHIVE,
        }
        # # Silver cleansed per analisi
        # {
        #     "path": "silver.fact_vendite",
        #     "layer": MedallionLayer.SILVER,
        #     "purpose": VisualizationPurpose.ANALYTICAL,
        #     "zorder_cols": ["DataOrdine", "CodiceCliente", "CodiceArticolo"],
        #     "partition_col": "Anno",
        # },
        # # Silver per operativo near real-time
        # {
        #     "path": "silver.eventi_produzione",
        #     "layer": MedallionLayer.SILVER,
        #     "purpose": VisualizationPurpose.OPERATIONAL,
        #     "zorder_cols": ["EventTimestamp", "MacchinarioID"],
        # },
        # # Gold per Power BI Direct Lake KPI
        # {
        #     "path": "gold.kpi_vendite_mensili",
        #     "layer": MedallionLayer.GOLD,
        #     "purpose": VisualizationPurpose.DASHBOARD_KPI,
        #     "zorder_cols": ["Anno", "Mese", "BusinessUnit"],
        # },
        # # Gold per export verso sistema esterno
        # {
        #     "path": "gold.export_fatturato",
        #     "layer": MedallionLayer.GOLD,
        #     "purpose": VisualizationPurpose.EXPORT,
        # },
        # # Silver feature store per ML
        # {
        #     "path": "silver.features_clienti",
        #     "layer": MedallionLayer.SILVER,
        #     "purpose": VisualizationPurpose.ML_TRAINING,
        # },
    ]

    # dry_run=True  → solo stampa le operazioni, non esegue nulla
    # dry_run=False → esegue OPTIMIZE/VACUUM e verifica le proprietà
    results = optimize_tables(spark, tables_to_optimize, dry_run=False, verify=True)

    print("\n" + "=" * 60)
    print("RIEPILOGO OTTIMIZZAZIONI E VERIFICHE")
    print("=" * 60)
    total_passed = 0
    total_failed = 0
    for r in results:
        print(f"\n {r['table']}")
        for op in r["operations"]:
            print(f"   → {op}")
        v = r.get("verification")
        if v:
            status_icon = "OK" if v["success"] else "KO"
            print(f"   {status_icon} Verifica: {v['passed']} OK, {v['failed']} KO")
            for chk in v["checks"]:
                if not chk.ok:
                    print(f"      {chk.prop}: atteso={chk.expected}, effettivo={chk.actual}")
            total_passed += v["passed"]
            total_failed += v["failed"]

    print("\n" + "=" * 60)
    print(f"TOTALE CHECK: {total_passed + total_failed}   {total_passed} OK   {total_failed} KO")
    print("=" * 60)

StatementMeta(, d0c1a2aa-900f-475a-a1c4-3169eac98b30, 3, Finished, Available, Finished, False)

2026-03-06 21:22:06,320 [INFO] Ottimizzazione: abfss://2cd8c8d6-210e-453e-a27d-453b9ffc21f1@onelake.dfs.fabric.microsoft.com/c3fcb4c1-59fb-408a-9972-b35608d80496/Tables/DB_Countries
2026-03-06 21:22:06,321 [INFO]   Profilo note: Bronze/Archive: nessun OPTIMIZE, retention lunga per audit.
2026-03-06 21:22:20,530 [INFO]   SET TBLPROPERTIES: {'delta.targetFileSize': '268435456', 'delta.dataSkippingNumIndexedCols': '32'}
2026-03-06 21:22:20,532 [INFO]   VACUUM RETAIN 720 HOURS



RIEPILOGO OTTIMIZZAZIONI E VERIFICHE

📦 abfss://2cd8c8d6-210e-453e-a27d-453b9ffc21f1@onelake.dfs.fabric.microsoft.com/c3fcb4c1-59fb-408a-9972-b35608d80496/Tables/DB_Countries
   → SET TBLPROPERTIES: {'delta.targetFileSize': '268435456', 'delta.dataSkippingNumIndexedCols': '32'}
   → VACUUM delta.`abfss://2cd8c8d6-210e-453e-a27d-453b9ffc21f1@onelake.dfs.fabric.microsoft.com/c3fcb4c1-59fb-408a-9972-b35608d80496/Tables/DB_Countries` RETAIN 720 HOURS
   ✅ Verifica: 2 OK, 0 KO

TOTALE CHECK: 2  ✅ 2 OK  ❌ 0 KO
